In [ ]:
# --- Install (einmalig pro Umgebung) ---
# %pip install -U leafmap[raster] rasterio requests numpy scipy geopandas shapely pysheds

from pathlib import Path
import os
import sys
import numpy as np
import rasterio
from rasterio import features
import rioxarray  # für spätere Flowtub-Erweiterungen
import requests
from scipy import ndimage
import leafmap
import time
import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output

# === SAVE ALL MASKS ===
OUTPUT_NODATA = -9999.0

# === CONFIGURATION ===
DEM_CHOICE = "COP30"           # "COP30" oder "NASADEM"
DEM_SOURCES = ["COP30", "NASADEM"]

# Mehrere Meeresspiegel-Anstiege in einer Schleife berechnen.
# 0.1 – 20m, fein aufgelöst im niedrigen Bereich (realistisch für Sturmfluten /
# mittelfristige Projektionen), gespreizter im hohen Bereich (extreme Szenarien).
# -> 20 Stufen * 4 Modelle * 2 DEMs = 160 Masken
# Bei Cache-Wiederverwendung (Remote-Webspace) ist der Erstlauf nach Upload lokal
# nur einmal nötig; auf Render werden die Masken nur per HTTP nachgeladen.
WATER_LEVELS = [0.1, 0.2, 0.3, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0,
                2.5, 3.0, 4.0, 5.0, 6.0, 7.5, 10.0, 12.5, 15.0, 20.0]

# Optional: alternativ eine Bereichs-Definition verwenden (auskommentiert lassen,
# wenn WATER_LEVELS genutzt wird).
# WATER_LEVEL_MIN = 0.1
# WATER_LEVEL_MAX = 100.5
# WATER_LEVEL_INCREMENT = 0.5

# Bounding Box Hamburg / Norddeutsche Küste
# europe: [-12.510040, 33.607112, 38.459671, 70.594009]
# netherlands [7.808984, 52.945352, 10.576295, 53.995902]
BBOX = (7.808984, 52.945352, 10.576295, 53.995902)  # Hamburg

# API Key: kommt ausschließlich aus der Umgebungsvariable.
#   lokal:   export OPENTOPOGRAPHY_API_KEY="<dein_key>"  (z.B. in ~/.zshenv)
#   Render:  im Dashboard unter Environment setzen (sync: false in render.yaml).
# Es gibt keinen Fallback im Quelltext mehr — der Key gehört nicht ins Repo.
# Voilà kann nicht interaktiv nachfragen; ohne Key bricht das Notebook
# beim Start mit einer klaren Fehlermeldung ab.
API_KEY = os.environ.get("OPENTOPOGRAPHY_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "OPENTOPOGRAPHY_API_KEY ist nicht gesetzt. "
        "Lokal: export OPENTOPOGRAPHY_API_KEY=... "
        "Render: Environment Variable im Dashboard setzen."
    )

# === ORDNERSTRUKTUR ===
DEM_DIR = Path("dem-src")          # hier liegen die "originalen" DEMs von OpenTopography
MASK_DIR = Path("flood-masks")     # hier landen die berechneten Masken (GeoTIFFs)
DEM_DIR.mkdir(parents=True, exist_ok=True)
MASK_DIR.mkdir(parents=True, exist_ok=True)

# === REMOTE-CACHE ===
# Wenn Masken lokal fehlen, werden sie von dieser URL nachgeladen, statt sie
# neu zu berechnen. Lege auf deinem Webspace unter dieser URL folgende Struktur an:
#   <REMOTE_BASE_URL>/dem-src/dem_COP30.tif
#   <REMOTE_BASE_URL>/dem-src/dem_NASADEM.tif
#   <REMOTE_BASE_URL>/flood-masks/flood_bathtub_COP30_5m.tif  (etc.)
#   <REMOTE_BASE_URL>/flood-masks/manifest.json
# Das Skript scripts/upload_artifacts.py erzeugt das manifest.json automatisch.
REMOTE_BASE_URL = os.environ.get(
    "FLOOD_REMOTE_BASE_URL",
    "",  # leer => kein Remote-Cache, alles wird lokal berechnet
).rstrip("/")
REMOTE_TIMEOUT = float(os.environ.get("FLOOD_REMOTE_TIMEOUT", "30"))

FILENAME_BASE = "flood"
FILENAME_SEPARATOR = "_"

# Welche Modelle werden berechnet / visualisiert?
# Flowtub nutzt eine an OpenStreetMap-Wasser angepasste Blau-Palette (#aad3df),
# damit die getönten Masken visuell zum Basemap-Wasser passen.
OSM_WATER_HEX = "#aad3df"

MODELS = {
    "bathtub":    {"colormap": "Reds",     "label": "Bathtub"},
    "connected":  {"colormap": "Greens",   "label": "Connected"},
    "iterative":  {"colormap": "Purples",  "label": "Iterative"},
    "flowtub":    {"colormap": "osm_water","label": "Flowtub"},   # ← NEU (OSM-Wasserblau)
}

# Eigene Matplotlib-Colormap für Flowtub: nur ein Bin, gefärbt mit OSM-Wasserblau.
# Wert 1 -> #aad3df, Wert 0 -> transparent, alles andere (Nodata) -> transparent.
try:
    from matplotlib.colors import LinearSegmentedColormap, ListedColormap
    _osm_cmap = ListedColormap([(0, 0, 0, 0), (0xaa/255, 0xd3/255, 0xdf/255, 0.85)])
    try:
        import matplotlib
        matplotlib.colormaps.register(_osm_cmap, name="osm_water")
    except Exception:
        pass
except Exception as _e:
    _osm_cmap = None
    print(f"WARN: osm_water Colormap konnte nicht registriert werden: {_e}")

start_time = time.time()

def time_elapsed():
    """Gibt die bisher vergangene Zeit als formatierten String aus und zurück."""
    delta_seconds = time.time() - start_time
    delta_time = str(datetime.timedelta(seconds=delta_seconds))
    print(f'Elapsed time: {delta_time}')
    return delta_time


def check_versions():
    print(f"--- Versions Check ---")
    print(f"Python: {sys.version}")
    print(f"Rasterio: {rasterio.__version__}")
    print(f"Numpy: {np.__version__}")
    if rasterio.__version__ < '1.5.0':
        print("WARNUNG: Rasterio < 1.5.0. Bitte 'conda install -c conda-forge rasterio>=1.5.0'.")
    print("----------------------\n")


check_versions()


# === DOWNLOAD DEM ===
def download_opentopography_dem(dataset, bbox, api_key, output_file, force=False):
    """
    Lädt ein DEM (COP30 oder NASADEM) für die gegebene BBOX und speichert es als GeoTIFF.
    output_file ist ein pathlib.Path. Bei force=True wird eine bestehende Datei überschrieben.
    """
    dataset = dataset.upper()
    if dataset not in {"COP30", "NASADEM"}:
        raise ValueError(f"dataset must be 'COP30' or 'NASADEM', got {dataset}")
    west, south, east, north = map(float, bbox)
    if west >= east or south >= north:
        raise ValueError("Invalid BBOX")
    endpoint = "https://portal.opentopography.org/API/globaldem"
    params = {
        "demtype": dataset, "south": south, "north": north,
        "west": west, "east": east, "outputFormat": "GTiff", "API_Key": api_key,
    }
    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    if output_file.exists() and not force:
        print(f"DEM already cached at {output_file}, skipping download.")
        return output_file
    print(f"Requesting {dataset} data... {time_elapsed()}")
    with requests.get(endpoint, params=params, stream=True, timeout=(30, 600)) as response:
        if not response.ok:
            raise RuntimeError(f"HTTP {response.status_code}: {response.text[:500]}")
        content_type = response.headers.get("Content-Type", "").lower()
        if any(x in content_type for x in ("text", "json", "xml")):
            raise RuntimeError(f"Not a GeoTIFF: {response.text[:500]}")
        with output_file.open("wb") as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)
    with rasterio.open(output_file) as src:
        print(f"Downloaded {src.width} × {src.height} pixels, CRS: {src.crs}")
    return output_file


# === TIER 1: Connected Components ===
def compute_connected_flood(dem_data, threshold, connectivity=4):
    """Nur Pixel, die mit dem DEM-Rand verbunden sind UND <= threshold."""
    valid = np.isfinite(dem_data)
    low = valid & (dem_data <= threshold)
    seeds = np.zeros_like(low, dtype=bool)
    seeds[0, :] = low[0, :]
    seeds[-1, :] = low[-1, :]
    seeds[:, 0] = low[:, 0]
    seeds[:, -1] = low[:, -1]
    structure = ndimage.generate_binary_structure(2, connectivity)
    labeled, n_components = ndimage.label(low, structure=structure)
    connected = np.zeros_like(low, dtype=bool)
    for label_id in range(1, n_components + 1):
        component = (labeled == label_id)
        if np.any(component & seeds):
            connected |= component
    return connected, n_components


# === TIER 2: Iterative Seed-Spread ===
def compute_connected_flood_iterative(dem_data, threshold, connectivity=4):
    """Iterative Variante – Ergebnis sollte Tier 1 entsprechen."""
    valid = np.isfinite(dem_data)
    low = valid & (dem_data <= threshold)
    seeds = np.zeros_like(low, dtype=bool)
    seeds[0, :] = low[0, :]
    seeds[-1, :] = low[-1, :]
    seeds[:, 0] = low[:, 0]
    seeds[:, -1] = low[:, -1]
    flood = seeds.copy()
    structure = ndimage.generate_binary_structure(2, connectivity)
    iteration = 0
    while True:
        prev = flood.copy()
        flood = (ndimage.binary_dilation(flood, structure=structure) & low) | seeds
        iteration += 1
        if np.array_equal(flood, prev):
            break
    print(f"  Iterative converged after {iteration} iterations")
    return flood


# === TIER 3: Flowtub (vom Meer ausgehender Fluss) ===
# Diese Funktion bleibt UNVERÄNDERT – wir wrappen sie nur beim Speichern,
# damit aus dem uint8-Array ein gültiger GeoTIFF wird.
def flowtub_simulation(dem_array, sea_level_rise):
    """
    Flowtub-Simulation: Wasser fließt vom Meer (DEM-Rand) aus in tiefer gelegene
    Pixel hinein. Dämme/Deiche, deren Höhe > sea_level_rise bleibt, werden nicht
    überwunden. Senken ohne Verbindung zum Meer bleiben trocken.

    Rückgabe: np.uint8 Maske (1 = geflutet, 0 = trocken) – gleiche Shape wie dem_array.
    """
    mask = (dem_array <= sea_level_rise)
    labeled_array, num_features = ndimage.label(mask)
    sea_label = labeled_array[0, 0]
    if sea_label == 0:
        return np.zeros_like(dem_array, dtype=np.uint8)
    return (labeled_array == sea_label).astype(np.uint8)


# === MASK SPEICHERN ===
# Akzeptiert sowohl bool- als auch 0/1-Arrays und schreibt einen gültigen GeoTIFF
# mit CRS/Transform/Shape aus dem DEM-Profil. Keine globalen Variablen mehr.
def save_mask(mask, filename, dem_profile):
    """
    Speichert eine Flood-Maske als GeoTIFF.

    mask       : 2D-Array (bool ODER 0/1).
    filename   : Zielpfad.
    dem_profile: rasterio-Profil des DEMs (liefert CRS, Transform, dtype, count, nodata).
    """
    mask = np.asarray(mask)
    if mask.dtype == bool:
        bool_mask = mask
    else:
        bool_mask = mask.astype(bool)
    out = np.full(mask.shape, OUTPUT_NODATA, dtype=np.float32)
    out[bool_mask] = 1.0
    profile = dem_profile.copy()
    profile.update(
        driver="GTiff",
        dtype="float32",
        count=1,
        nodata=OUTPUT_NODATA,
        compress="deflate",
    )
    filename = Path(filename)
    filename.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(filename, "w", **profile) as dst:
        dst.write(out, 1)
    return filename


def get_mask_filename(dem_provider, model, sea_level_m, base=MASK_DIR):
    """Liefert den kanonischen Dateinamen für eine Maske."""
    return base / f"{FILENAME_BASE}{FILENAME_SEPARATOR}{model}{FILENAME_SEPARATOR}{dem_provider}{FILENAME_SEPARATOR}{sea_level_m:g}m.tif"


def get_dem_filename(dem_provider):
    return DEM_DIR / f"dem_{dem_provider}.tif"


# === Wasserpegel-Liste aufbereiten ===
def get_water_levels():
    """Gibt die zu berechnenden Meeresspiegel als Liste zurück."""
    if 'WATER_LEVELS' in globals() and WATER_LEVELS:
        return list(WATER_LEVELS)
    return list(np.arange(WATER_LEVEL_MIN, WATER_LEVEL_MAX, WATER_LEVEL_INCREMENT))



# === REMOTE-CACHE: Funktionen ===
def _remote_url_exists(url, timeout=REMOTE_TIMEOUT):
    """HEAD-Request, um zu prüfen ob eine Remote-Datei existiert."""
    if not url:
        return False
    try:
        r = requests.head(url, allow_redirects=True, timeout=timeout)
        if r.status_code == 405:  # Server unterstützt kein HEAD -> GET probieren
            r = requests.get(url, stream=True, timeout=timeout)
            exists = r.ok
            r.close()
            return exists
        return r.ok
    except requests.RequestException as e:
        print(f"  [remote] HEAD fehlgeschlagen für {url}: {e}")
        return False


def _remote_fetch(url, dest_path, timeout=REMOTE_TIMEOUT):
    """Datei via HTTP GET herunterladen, atomar (in tmp, dann rename)."""
    dest_path = Path(dest_path)
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = dest_path.with_suffix(dest_path.suffix + ".part")
    try:
        with requests.get(url, stream=True, timeout=timeout) as r:
            r.raise_for_status()
            with tmp_path.open("wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 256):
                    if chunk:
                        f.write(chunk)
        tmp_path.replace(dest_path)
        return True
    except requests.RequestException as e:
        print(f"  [remote] Download fehlgeschlagen {url}: {e}")
        if tmp_path.exists():
            try:
                tmp_path.unlink()
            except OSError:
                pass
        return False


def ensure_local_from_remote(local_path, relative_key):
    """
    Stellt sicher, dass `local_path` lokal existiert. Wenn nicht und ein
    Remote-Cache konfiguriert ist, wird die Datei heruntergeladen.
    Rückgabe: True wenn die Datei danach lokal vorhanden ist.
    """
    local_path = Path(local_path)
    if local_path.exists():
        return True
    if not REMOTE_BASE_URL:
        return False
    url = f"{REMOTE_BASE_URL}/{relative_key.lstrip(chr(47))}"
    print(f"  [remote] lade {relative_key} ...")
    return _remote_fetch(url, local_path)


# === HAUPTSCHLEIFE: DEM laden + alle 4 Modelle rechnen ===
for dem in DEM_SOURCES:
    dem_filename = get_dem_filename(dem)
    print(f"\n========== DEM: {dem} ==========")

    # 0) DEM aus Remote-Cache laden, falls lokal fehlt
    if not dem_filename.exists():
        rel = f"dem-src/{dem_filename.name}"
        if ensure_local_from_remote(dem_filename, rel):
            print(f"  DEM {dem} aus Remote-Cache geladen: {dem_filename}")

    # 1) DEM herunterladen (oder Cache nutzen)
    dem_file = download_opentopography_dem(
        dataset=dem, bbox=BBOX, api_key=API_KEY, output_file=dem_filename
    )

    # 2) DEM in den Speicher lesen + Profil merken
    with rasterio.open(dem_file) as src:
        dem_raster = src.read(1, masked=True).astype(np.float32)
        dem_profile = src.profile.copy()
        dem_transform = src.transform
        dem_crs = src.crs
    dem_data = dem_raster.filled(np.nan)
    valid = np.isfinite(dem_data)
    valid_elevations = dem_data[valid]
    print(f"DEM: {dem} | shape={dem_data.shape} | CRS={dem_crs} | "
          f"Min={np.min(valid_elevations):.2f}m | Max={np.max(valid_elevations):.2f}m | "
          f"Mean={np.mean(valid_elevations):.2f}m")

    # 3) Alle Wasserstände durchgehen
    for rise in get_water_levels():
        print(f" {time_elapsed}\n  -- Sea-level rise = {rise:g} m --")

        # --- Tier 0: Bathtub ---
        bathtub_path = get_mask_filename(dem, "bathtub", rise)
        rel_bathtub = f"flood-masks/{bathtub_path.name}"
        if bathtub_path.exists():
            print(f"  bathtub mask exists @ {bathtub_path}, skipping compute")
        elif ensure_local_from_remote(bathtub_path, rel_bathtub):
            print(f"  bathtub mask aus Remote-Cache geladen")
        else:
            bathtub_mask = valid & (dem_data <= rise)
            print(f"  Bathtub: {np.count_nonzero(bathtub_mask):,} cells")
            save_mask(bathtub_mask, bathtub_path, dem_profile)

        # --- Tier 1: Connected Components ---
        connected_path = get_mask_filename(dem, "connected", rise)
        rel_connected = f"flood-masks/{connected_path.name}"
        if connected_path.exists():
            print(f"  connected mask exists @ {connected_path}, skipping compute")
        elif ensure_local_from_remote(connected_path, rel_connected):
            print(f"  connected mask aus Remote-Cache geladen")
        else:
            connected_mask, n_comp = compute_connected_flood(dem_data, rise, connectivity=4)
            bathtub_count = np.count_nonzero(valid & (dem_data <= rise))
            print(f"  Connected: {np.count_nonzero(connected_mask):,} cells "
                  f"(removed {bathtub_count - np.count_nonzero(connected_mask):,} isolated cells)")
            save_mask(connected_mask, connected_path, dem_profile)

        # --- Tier 2: Iterative Seed-Spread ---
        iterative_path = get_mask_filename(dem, "iterative", rise)
        rel_iterative = f"flood-masks/{iterative_path.name}"
        if iterative_path.exists():
            print(f"  iterative mask exists @ {iterative_path}, skipping compute")
        elif ensure_local_from_remote(iterative_path, rel_iterative):
            print(f"  iterative mask aus Remote-Cache geladen")
        else:
            iterative_mask = compute_connected_flood_iterative(dem_data, rise, connectivity=4)
            print(f"  Iterative: {np.count_nonzero(iterative_mask):,} cells")
            save_mask(iterative_mask, iterative_path, dem_profile)

        # --- Tier 3: Flowtub (NEU) ---
        flowtub_path = get_mask_filename(dem, "flowtub", rise)
        rel_flowtub = f"flood-masks/{flowtub_path.name}"
        if flowtub_path.exists():
            print(f"  flowtub mask exists @ {flowtub_path}, skipping compute")
        elif ensure_local_from_remote(flowtub_path, rel_flowtub):
            print(f"  flowtub mask aus Remote-Cache geladen")
        else:
            # flowtub_simulation nimmt das DEM-Array und liefert uint8 (0/1).
            # Wir konvertieren in bool und speichern wie die anderen Modelle.
            flowtub_uint8 = flowtub_simulation(dem_data, rise)
            flowtub_mask = flowtub_uint8.astype(bool)
            print(f"  Flowtub: {np.count_nonzero(flowtub_mask):,} cells")
            save_mask(flowtub_mask, flowtub_path, dem_profile)

    print(f"\nDone for {dem} at {time_elapsed()}")

print("\n========== Berechnung abgeschlossen ==========")
time_elapsed()


# === LEAFMAP MIT WIDGETS ===
west, south, east, north = BBOX
center = [(south + north) / 2, (west + east) / 2]

m = leafmap.Map(center=center, zoom=8)
m.add_basemap("HYBRID")

# Welche DEM-Datei wird gerade angezeigt?
current_dem = {DEM_CHOICE}

# Meeresspiegel-Slider: kleiner, prüfbarer Bereich (0.5 ... max der berechneten Werte).
available_levels = sorted({lvl for lvl in get_water_levels()})
slider_min = min(available_levels)
slider_max = max(available_levels)
slider_step = 0.5 if len(available_levels) > 1 else 0.1
slider_value = available_levels[0]

slider = widgets.FloatSlider(
    value=slider_value,
    min=slider_min,
    max=slider_max,
    step=slider_step,
    description='Sea level (m):',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.1f',
)

dem_dropdown = widgets.Dropdown(
    options=[(name, name) for name in DEM_SOURCES],
    value=DEM_CHOICE,
    description='DEM:',
)


def _layer_name(model, dem_name, sea_level):
    return f"{MODELS[model]['label']} ({dem_name}, ≤{sea_level:g}m)"


def _layer_names_in_usev(map_obj):
    """Gibt die Namen aller Layer auf der Karte zurück (robust gegen Listen von Objekten)."""
    layers = getattr(map_obj, "layers", []) or []
    names = set()
    for layer in layers:
        name = getattr(layer, "name", None)
        if name is None and isinstance(layer, str):
            name = layer
        if name:
            names.add(name)
    return names


def _remove_model_layers(map_obj):
    """Entfernt alle bisherigen Modell-Layer + DEM-Layer aus der Karte."""
    labels = {meta["label"] for meta in MODELS.values()}
    for layer_name in list(_layer_names_in_use(map_obj)):
        is_model = any(layer_name.startswith(f"{label} (") for label in labels)
        is_dem = layer_name.startswith("DEM: ")
        if is_model or is_dem:
            try:
                map_obj.remove_layer(layer_name)
            except Exception:
                pass


def _refresh_layers(map_obj, dem_name, sea_level):
    """DEM-Layer + 4 Modell-Layer für die aktuelle Auswahl neu zeichnen."""
    # alte Modell-Layer weg
    _remove_model_layers(map_obj)

    dem_path = get_dem_filename(dem_name)
    if dem_path.exists():
        try:
            map_obj.remove_layer(f"DEM: {dem_name}")
        except Exception:
            pass
        map_obj.add_raster(
            str(dem_path),
            colormap="terrain",
            layer_name=f"DEM: {dem_name}",
            opacity=0.4,
        )

    # nächstgelegenen berechneten Wasserstand suchen (für Demo: nur einer da)
    available = sorted(get_water_levels())
    nearest = min(available, key=lambda x: abs(x - sea_level))
    for model, meta in MODELS.items():
        mask_path = get_mask_filename(dem_name, model, nearest)
        if not mask_path.exists():
            print(f"⚠️  Maske fehlt: {mask_path}")
            continue
        # Maske ist 0/1: nur Wert 1 bekommt Farbe, Nodata bleibt transparent.
        kwargs = dict(
            vmin=0, vmax=1,
            nodata=OUTPUT_NODATA,
            layer_name=_layer_name(model, dem_name, nearest),
            opacity=0.75,
        )
        if meta["colormap"] == "osm_water":
            kwargs["colormap"] = _osm_cmap
        else:
            kwargs["colormap"] = meta["colormap"]
        map_obj.add_raster(str(mask_path), **kwargs)


def _on_change(change):
    if change["name"] in ("value", "new"):
        clear_output(wait=True)
        display(ui_box)
        display(m)
        _refresh_layers(m, dem_dropdown.value, slider.value)


slider.observe(_on_change, names="value")
dem_dropdown.observe(_on_change, names="value")

ui_box = widgets.HBox([dem_dropdown, slider])
display(ui_box)

# initial zeichnen
_refresh_layers(m, dem_dropdown.value, slider.value)
m
